In [0]:
# ============================================================
# NOTEBOOK 2: SILVER LAYER
# Team AryaBit | HackBricks 2026
# Purpose: Clean data + Engineer features for ML
# Key Features: Grade delta, Absenteeism trend, Financial stress index
# ============================================================

import logging
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    when, col, round as spark_round,
    lit, current_timestamp
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AryaBit.Silver")

# ── CONFIG ───────────────────────────────────────────────────
CATALOG       = "workspace"
SCHEMA        = "default"
BRONZE_TABLE  = f"{CATALOG}.{SCHEMA}.bronze_student_dropout"
SILVER_TABLE  = f"{CATALOG}.{SCHEMA}.silver_student_dropout"

# ── STEP 1: LOAD BRONZE ──────────────────────────────────────
def load_bronze():
    logger.info("Loading Bronze table...")
    df = spark.table(BRONZE_TABLE)
    logger.info(f"Bronze record count: {df.count()}")
    return df

# ── STEP 2: CLEAN DATA ───────────────────────────────────────
def clean_data(df):
    logger.info("Cleaning data...")
    
    # Drop metadata columns from bronze
    df = df.drop("_ingestion_timestamp", "_source_file", "_layer")
    
    # Create binary target: Dropout=1, Enrolled/Graduate=0
    df = df.withColumn(
        "dropout_label",
        when(col("target") == "Dropout", 1).otherwise(0)
    )
    
    # Null check
    null_counts = {c: df.filter(col(c).isNull()).count() for c in df.columns}
    nulls_found = {k: v for k, v in null_counts.items() if v > 0}
    if nulls_found:
        logger.warning(f"Nulls found: {nulls_found}")
        df = df.dropna()
    else:
        logger.info("No nulls found ✅")
    
    return df

# ── STEP 3: FEATURE ENGINEERING ──────────────────────────────
def engineer_features(df):
    logger.info("Engineering features...")

    # ── FEATURE 1: Semester-on-semester GRADE DELTA ──────────
    # How much did the student's grade change between sem1 and sem2?
    # Negative delta = declining performance = risk signal
    df = df.withColumn(
        "grade_delta",
        spark_round(
            col("curricular_units_2nd_sem_grade") -
            col("curricular_units_1st_sem_grade"), 4
        )
    )

    # ── FEATURE 2: APPROVAL RATE per semester ────────────────
    # Units approved / units enrolled — measures academic engagement
    df = df.withColumn(
        "approval_rate_sem1",
        when(
            col("curricular_units_1st_sem_enrolled") > 0,
            spark_round(
                col("curricular_units_1st_sem_approved") /
                col("curricular_units_1st_sem_enrolled"), 4
            )
        ).otherwise(0.0)
    )

    df = df.withColumn(
        "approval_rate_sem2",
        when(
            col("curricular_units_2nd_sem_enrolled") > 0,
            spark_round(
                col("curricular_units_2nd_sem_approved") /
                col("curricular_units_2nd_sem_enrolled"), 4
            )
        ).otherwise(0.0)
    )

    # ── FEATURE 3: ABSENTEEISM TREND ─────────────────────────
    # Drop in approval rate from sem1 to sem2
    # Captures disengagement trajectory
    df = df.withColumn(
        "absenteeism_trend",
        spark_round(
            col("approval_rate_sem2") - col("approval_rate_sem1"), 4
        )
    )

    # ── FEATURE 4: FINANCIAL STRESS INDEX ────────────────────
    # Composite score: debtor + tuition not up to date + no scholarship
    # Range: 0 (no stress) to 3 (maximum stress)
    df = df.withColumn(
        "financial_stress_index",
        col("debtor") +
        (1 - col("tuition_fees_up_to_date")) +
        (1 - col("scholarship_holder"))
    )

    # ── FEATURE 5: ACADEMIC LOAD RATIO ───────────────────────
    # Total evaluations vs total enrolled
    # Overloaded students with poor performance = high risk
    df = df.withColumn(
        "total_enrolled",
        col("curricular_units_1st_sem_enrolled") +
        col("curricular_units_2nd_sem_enrolled")
    ).withColumn(
        "total_approved",
        col("curricular_units_1st_sem_approved") +
        col("curricular_units_2nd_sem_approved")
    ).withColumn(
        "overall_approval_rate",
        when(
            col("total_enrolled") > 0,
            spark_round(col("total_approved") / col("total_enrolled"), 4)
        ).otherwise(0.0)
    )

    # ── FEATURE 6: AVERAGE GRADE OVERALL ─────────────────────
    df = df.withColumn(
        "avg_grade_overall",
        spark_round(
            (col("curricular_units_1st_sem_grade") +
             col("curricular_units_2nd_sem_grade")) / 2, 4
        )
    )

    logger.info("Feature engineering complete ✅")
    return df

# ── STEP 4: ADD SILVER METADATA ──────────────────────────────
def add_silver_metadata(df):
    return (
        df
        .withColumn("_silver_timestamp", current_timestamp())
        .withColumn("_layer", lit("silver"))
    )

# ── STEP 5: WRITE SILVER TABLE ───────────────────────────────
def write_silver_table(df, table_name: str):
    logger.info(f"Writing Silver Delta table: {table_name}")
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    logger.info(f"Silver table written: {table_name}")

# ── STEP 6: VALIDATE FEATURES ────────────────────────────────
def validate_features(table_name: str):
    print("\n" + "="*60)
    print("SILVER LAYER — FEATURE VALIDATION")
    print("="*60)

    spark.sql(f"""
        SELECT
            ROUND(AVG(grade_delta), 4)            as avg_grade_delta,
            ROUND(AVG(absenteeism_trend), 4)      as avg_absenteeism_trend,
            ROUND(AVG(financial_stress_index), 4) as avg_financial_stress,
            ROUND(AVG(overall_approval_rate), 4)  as avg_approval_rate,
            ROUND(AVG(avg_grade_overall), 4)      as avg_grade_overall
        FROM {table_name}
    """).show()

    print("\nFEATURES BY DROPOUT LABEL:")
    spark.sql(f"""
        SELECT
            dropout_label,
            COUNT(*)                                    as count,
            ROUND(AVG(grade_delta), 4)                 as avg_grade_delta,
            ROUND(AVG(absenteeism_trend), 4)           as avg_absenteeism_trend,
            ROUND(AVG(financial_stress_index), 4)      as avg_financial_stress,
            ROUND(AVG(overall_approval_rate), 4)       as avg_approval_rate
        FROM {table_name}
        GROUP BY dropout_label
        ORDER BY dropout_label
    """).show()

    print("\nRECORD COUNT:")
    count = spark.sql(
        f"SELECT COUNT(*) as total FROM {table_name}"
    ).collect()[0][0]
    print(f"Total silver records: {count}")

# ── MAIN EXECUTION ────────────────────────────────────────────
def run_silver_pipeline():
    logger.info("Starting Silver Layer Pipeline — AryaBit")
    try:
        # Load bronze
        bronze_df = load_bronze()

        # Clean
        clean_df = clean_data(bronze_df)

        # Engineer features
        featured_df = engineer_features(clean_df)

        # Add metadata
        silver_df = add_silver_metadata(featured_df)

        # Write
        write_silver_table(silver_df, SILVER_TABLE)

        # Validate
        validate_features(SILVER_TABLE)

        logger.info("Silver Layer COMPLETE ✅")
        return True

    except Exception as e:
        logger.error(f"Silver Pipeline FAILED: {str(e)}")
        raise e

# ── RUN ───────────────────────────────────────────────────────
run_silver_pipeline()